[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ohtopico/site/blob/main/notebooks/wikipedia/03-deteccao-de-picos.ipynb)

# Detecção de picos: quando o interesse dispara

Notícias, lançamentos e datas comemorativas fazem as visitas de um verbete dispararem. Neste notebook você cria um detector de picos com estatística simples (mediana e desvio robusto), ajusta a sensibilidade e investiga o que aconteceu em cada data.

**Público:** professores, 9º ano, ensino médio e cursos técnicos · **Tempo:** 1 a 2 aulas de 50 min · **BNCC:** EF09MA21, EF08CO11, EF09CO10

Notebook de [Júlia Roberta Quoos Alves](https://ohtopico.com.br/curriculo/) · [ohtopico.com.br](https://ohtopico.com.br/interesse-por-ia/notebooks/) · Licença CC BY-NC-SA 4.0

**Como usar:** clique em cada célula de código e aperte o botão de play (ou Shift + Enter), de cima para baixo. Os campos ao lado das células (listas e caixas de texto) podem ser trocados à vontade: depois de mudar, rode a célula de novo.

## Objetivos
- Usar a mediana de uma janela móvel como referência do que é "normal".
- Medir o quanto um valor se afasta do normal e definir um limite para chamar de pico.
- Checar as explicações dos picos em fontes confiáveis.

In [ ]:
# @title Configuração (rode primeiro) { display-mode: "form" }
# Funções para baixar os dados públicos da Wikimedia. Não precisa mudar nada aqui.
import json, re, time, urllib.parse
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

CORES = ['#157f3b', '#b3123a', '#c99a00', '#111111', '#5fb27c', '#7a1f33']
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.prop_cycle': plt.cycler(color=CORES), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3, 'font.size': 11})
pd.options.display.float_format = '{:,.0f}'.format
UA = {'User-Agent': 'notebooks-ohtopico/1.0 (https://ohtopico.com.br; contato@ohtopico.com.br)'}
FONTE = 'Fonte: Wikimedia, visitas de pessoas à Wikipédia em português'


def _get(url, params=None):
    r = requests.get(url, params=params, headers=UA, timeout=60)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()


def titulo_oficial(titulo):
    """Corrige maiúsculas e segue redirecionamentos: 'ia' vira 'Inteligência artificial'."""
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', titles=titulo, redirects=1, format='json', formatversion=2))
    pagina = j['query']['pages'][0]
    if pagina.get('missing'):
        raise ValueError(f'Não achei o verbete "{titulo}" na Wikipédia em português. Confira a grafia.')
    return pagina['title']


def visitas(titulos, inicio='2016-01-01', fim=None, por='mes'):
    """Tabela com as visitas de cada verbete. por='mes' ou por='dia'."""
    if isinstance(titulos, str):
        titulos = [titulos]
    fim = pd.Timestamp(fim) if fim else pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    if por == 'mes':
        fim = fim.replace(day=1) - pd.Timedelta(days=1)  # só meses completos
    colunas = {}
    for t in titulos:
        t = titulo_oficial(t)
        art = urllib.parse.quote(t.replace(' ', '_'), safe='')
        g = 'monthly' if por == 'mes' else 'daily'
        url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/pt.wikipedia/all-access/user/{art}/{g}/'
               f'{pd.Timestamp(inicio):%Y%m%d}00/{fim:%Y%m%d}00')
        j = _get(url) or {'items': []}
        colunas[t] = pd.Series({pd.Timestamp(i['timestamp'][:8]): i['views'] for i in j['items']}, dtype='float64')
        time.sleep(.1)
    df = pd.DataFrame(colunas).sort_index()
    idx = pd.date_range(pd.Timestamp(inicio), fim, freq='MS' if por == 'mes' else 'D')
    df = df.reindex(idx).fillna(0).astype(int)
    df.index.name = 'mês' if por == 'mes' else 'dia'
    return df


def texto(titulo):
    """Texto puro de um verbete (sem tabelas nem imagens)."""
    t = titulo_oficial(titulo)
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', prop='extracts', explaintext=1, titles=t,
                                                         format='json', formatversion=2))
    return j['query']['pages'][0].get('extract', '')


def fmt(v, casas=0):
    """Número no formato brasileiro: 1.234,5"""
    return f'{v:,.{casas}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


def baixar_csv(df, nome):
    df.to_csv(nome)
    try:
        from google.colab import files
        files.download(nome)
    except ImportError:
        print('Arquivo salvo:', nome)


print('Pronto. Funções disponíveis: visitas(), texto(), titulo_oficial(), fmt(), baixar_csv().')

## Passo 1. Visitas por dia

In [ ]:
verbete = 'Copa do Mundo FIFA'  # @param {type:"string"}
dias = 730  # @param {type:"slider", min:90, max:1500, step:30}
inicio = pd.Timestamp.today().normalize() - pd.Timedelta(days=dias)
d = visitas(verbete, inicio=inicio, por='dia').iloc[:, 0]
ax = d.plot(linewidth=1); ax.set_title(f'{d.name}: visitas por dia'); ax.set_ylabel('Visitas'); plt.show()

## Passo 2. O que é normal?
Para cada dia, olhamos os dias ao redor (uma **janela**) e calculamos a **mediana**. A mediana quase não muda com um pico isolado, por isso é uma boa referência. Depois medimos o **desvio robusto** (a mediana das distâncias até a mediana) e dizemos quantos desvios cada dia está acima do normal. Esse número é a **pontuação** do dia.

In [ ]:
def detectar(serie, janela=29, limiar=6):
    normal = serie.rolling(janela, center=True, min_periods=janela // 2).median()
    desvio = (serie - normal).abs().rolling(janela, center=True, min_periods=janela // 2).median() * 1.4826
    pontuacao = (serie - normal) / desvio.clip(lower=max(1, serie.median() * .05))
    return normal, pontuacao, pontuacao > limiar

@interact(janela=widgets.IntSlider(value=29, min=7, max=91, step=2, description='janela (dias)'),
          limiar=widgets.FloatSlider(value=6, min=2, max=20, step=.5, description='limiar'))
def ver(janela, limiar):
    normal, pont, pico = detectar(d, janela, limiar)
    ax = d.plot(linewidth=1, color='#4a4a4a', label='visitas')
    normal.plot(ax=ax, linewidth=2, color=CORES[0], label='normal (mediana da janela)')
    ax.scatter(d.index[pico], d[pico], color=CORES[1], zorder=3, s=30, label=f'picos ({pico.sum()} dias)')
    ax.set_title('Picos encontrados'); ax.legend(); plt.show()

**Para discutir:** o que acontece com o número de picos quando o limiar diminui? E quando a janela aumenta? Não existe limiar certo: é uma escolha, e toda escolha deve ser explicada.

## Passo 3. A lista de picos
Agrupamos dias de pico seguidos em um só evento e ordenamos pelo tamanho.

In [ ]:
janela = 29  # @param {type:"integer"}
limiar = 6  # @param {type:"number"}
normal, pont, pico = detectar(d, janela, limiar)
evento = (pico != pico.shift()).cumsum()[pico]
picos = (pd.DataFrame({'dia': d.index[pico], 'visitas': d[pico].values, 'normal': normal[pico].values.round(), 'evento': evento.values})
         .sort_values('visitas', ascending=False).drop_duplicates('evento').drop(columns='evento'))
picos['vezes o normal'] = (picos['visitas'] / picos['normal'].clip(lower=1)).round(1)
picos['dia'] = picos['dia'].dt.strftime('%d/%m/%Y')
picos.head(10).reset_index(drop=True)

## Passo 4. Investigue
Para cada um dos três maiores picos, procure em duas fontes confiáveis (um jornal e um site oficial, por exemplo) o que aconteceu naquele dia ou na véspera. Anote a fonte e a data de cada explicação.

| Dia | Explicação | Fonte 1 | Fonte 2 |
|---|---|---|---|
|  |  |  |  |

Lembre: os dados mostram **quando** o interesse mudou. O **porquê** vem das fontes.

## Sua vez
Compare dois verbetes ligados (por exemplo, *Eclipse* e *Lua*). Os picos acontecem nos mesmos dias? Escreva o código abaixo.

In [ ]:
# escreva aqui

<details><summary><b>Ver uma solução</b></summary>

```python
dois = visitas(['Eclipse', 'Lua'], inicio=inicio, por='dia')
marcados = pd.DataFrame({c: detectar(dois[c])[2] for c in dois})
print('Dias de pico em comum:', list(dois.index[marcados.all(axis=1)].strftime('%d/%m/%Y')))
```
</details>

## Para fechar
Detectores de anomalia como este são usados para achar fraudes em cartões, falhas em máquinas e surtos de doenças. A ideia é a mesma: definir o normal, medir a distância e escolher um limite.